In [ ]:
import torch
import torch.nn as nn

In [ ]:
import pandas as pd

dst = pd.read_csv("data.csv")
dst.head()

,0
0,535+278=813
1,437+985=1422
2,70+959=1029
3,763+355=1118
4,3+514=517


In [ ]:
dst.dtypes

,0
0,object


In [ ]:
from sklearn.model_selection import train_test_split

train_dst, test_dst = train_test_split(
    dst,
    test_size=0.2,
    random_state=42
)

In [ ]:
len(train_dst)

15830

In [ ]:
len(test_dst)

3958

In [ ]:
train_dst

,0
17982,161+443=604
2640,154+130=284
222,390+460=850
18528,630+691=1321
4318,333+226=559
...,...
11284,767+49=816
11964,593+224=817
5390,289+728=1017
860,408+820=1228


In [ ]:
symbol2id = {}
id2symbol = {}

for i in range(10):
    symbol2id[str(i)] = i
    id2symbol[i] = str(i)

EQUAL_ID = 10
symbol2id["="] = EQUAL_ID
id2symbol[EQUAL_ID] = "="

PLUS_ID = 11
symbol2id["+"] = PLUS_ID
id2symbol[PLUS_ID] = "+"

EOS_ID = 12
symbol2id["<EOS>"] = EOS_ID
id2symbol[EOS_ID] = "<EOS>"

PAD_ID = 13
symbol2id["<PAD>"] = PAD_ID
id2symbol[PAD_ID] = "<PAD>"


In [ ]:
len(symbol2id)

14

In [ ]:
from typing import List

In [ ]:
def encode(example: str) -> List[int]:
    result = []

    for symbol in example:
        result.append(symbol2id[symbol])

    if example[-1] != "=":
        result.append(EOS_ID)

    return result

def decode(example: torch.Tensor) -> str:
    result = ""
    for symbol in example:
        result += id2symbol[symbol.item()]

    return result



In [ ]:
train_dst

,0
17982,161+443=604
2640,154+130=284
222,390+460=850
18528,630+691=1321
4318,333+226=559
...,...
11284,767+49=816
11964,593+224=817
5390,289+728=1017
860,408+820=1228


In [ ]:
def collate_fn(examples: List[str]):
    encoded_examples = []
    targets = []
    max_length = 0

    for example in examples:
        if len(example) > max_length:
            max_length = len(example)

    max_length += 1

    for example in examples:
        enc_ex = encode(example)
        enc_ex.extend([PAD_ID] * (max_length - len(enc_ex)))
        encoded_examples.append(
            torch.tensor(enc_ex, dtype=torch.int))

        equal_id = enc_ex.index(EQUAL_ID)
        target = enc_ex.copy()
        target[:equal_id+1] = [PAD_ID] * (equal_id + 1)
        targets.append(
            torch.tensor(target, dtype= torch.long))


    return (torch.stack(encoded_examples, dim=0),
            torch.stack(targets, dim=0))


In [ ]:
def collate_fn_ab_ba(examples: List[str]):
    encoded_examples_ab = []
    encoded_examples_ba = []
    targets = []
    max_length = 0

    for example in examples:
        if len(example) > max_length:
            max_length = len(example)

    max_length += 1

    for example in examples:
        enc_ex = encode(example)
        enc_ex.extend([PAD_ID] * (max_length - len(enc_ex)))
        encoded_examples_ab.append(
            torch.tensor(enc_ex, dtype=torch.int))

        equal_id = enc_ex.index(EQUAL_ID)
        plus_id = enc_ex.index(PLUS_ID)
        enc_ex_ab = enc_ex.copy()
        enc_ex_ab[:plus_id], enc_ex_ab[plus_id+1:equal_id] = enc_ex_ab[plus_id+1:equal_id], enc_ex_ab[:plus_id]
        encoded_examples_ba.append(
            torch.tensor(enc_ex_ab, dtype=torch.int))

        target = enc_ex.copy()
        target[:equal_id+1] = [PAD_ID] * (equal_id + 1)
        targets.append(
            torch.tensor(target, dtype= torch.long))


    return (torch.stack(encoded_examples_ab, dim=0),
            torch.stack(encoded_examples_ba, dim=0),
            torch.stack(targets, dim=0))


In [ ]:
import torch.nn.functional as F

def train_commutative_function(model, dataloader, loss_fn, optimizer,
                               lam:int = 0.05):

    model.train()

    losses = []
    accuracies = []
    exact_matches = []

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for inputs_id_ab, inputs_id_ba, targets in dataloader:
        inputs_id_ab = inputs_id_ab.to(device)
        inputs_id_ba = inputs_id_ba.to(device)
        targets = targets.to(device)
        targets = targets[:, 1:]

        _, outputs_ab = model(inputs_id_ab)
        outputs_ab = outputs_ab[:,:-1]
        loss_ab = loss_fn(outputs_ab.permute(0, 2, 1), targets)

        _, outputs_ba = model(inputs_id_ba)
        outputs_ba = outputs_ba[:,:-1]
        loss_ba = loss_fn(outputs_ba.permute(0, 2, 1), targets)

        logp_ab = F.log_softmax(outputs_ab, dim=-1)  # (B, T, V)
        logp_ba = F.log_softmax(outputs_ba, dim=-1)

        mask = (targets != PAD_ID)                  # (B, T)
        mask3 = mask.unsqueeze(-1).float()          # (B, T, 1)

        diff2 = (logp_ab - logp_ba) ** 2            # (B, T, V) MSE
        comm_loss = (diff2 * mask3).sum() / (mask3.sum() * outputs_ab.size(-1))
        loss = loss_ab + loss_ba + lam * comm_loss


        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        outputs = outputs_ab.argmax(-1)
        mask = (targets != PAD_ID)
        corrects = (outputs == targets) & mask
        accuracy = corrects.sum().item() / mask.sum().item()

        #accuracies.append((outputs.argmax(-1) == targets).float().mean().item())
        #accuracy
        accuracies.append(accuracy)
        losses.append(loss.item())

        #exact match
        pad_mask = (targets == PAD_ID)
        token_ok = (outputs == targets) | pad_mask
        exact_per_sample = token_ok.all(dim=1)
        exact_match = exact_per_sample.float().mean().item()
        exact_matches.append(exact_match)


    loss = np.mean(losses)
    accuracy = np.mean(accuracies)
    exact_match = np.mean(exact_matches)

    return loss, accuracy, exact_match

In [ ]:
collate_fn(['773+864=1234'])

(tensor([[ 7,  7,  3, 11,  8,  6,  4, 10,  1,  2,  3,  4, 12]],
        dtype=torch.int32),
 tensor([[13, 13, 13, 13, 13, 13, 13, 13,  1,  2,  3,  4, 12]]))

In [ ]:
collate_fn_ab_ba(['773+864=1234'])

(tensor([[ 7,  7,  3, 11,  8,  6,  4, 10,  1,  2,  3,  4, 12]],
        dtype=torch.int32),
 tensor([[ 8,  6,  4, 11,  7,  7,  3, 10,  1,  2,  3,  4, 12]],
        dtype=torch.int32),
 tensor([[13, 13, 13, 13, 13, 13, 13, 13,  1,  2,  3,  4, 12]]))

In [ ]:
from torch.utils.data import Dataset

class DatasetSum(Dataset):
    def __init__(self, data: List[str]):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


In [ ]:
class RNNAddition(nn.Module):

    def __init__(self, vocab_size:int, hidden_size:int):
        super().__init__()
        self.emb_layer = nn.Embedding(vocab_size, hidden_size)
        self.hidden_size = hidden_size
        self.W = nn.Linear(2*hidden_size,hidden_size)
        self.O = nn.Linear(hidden_size, vocab_size)

    def forward(self, inputs, h_0 = None):
        batch_size, seq_len = inputs.shape
        outputs = []
        inputs = self.emb_layer(inputs)

        if h_0 is None:
            h_t = torch.zeros(batch_size, self.hidden_size, device = inputs.device)
        else:
            h_t = h_0

        for i in range(seq_len):
            x_t = inputs[:,i,:]
            h_t = torch.tanh(self.W(torch.cat([x_t, h_t], dim = -1)))
            output = self.O(h_t)
            outputs.append(output)

        return h_t, torch.stack(outputs, dim= 1)


In [ ]:
@torch.no_grad()

def evaluation(model, dataloader, loss_fn):
  model.eval()

  losses = []
  accuracies = []
  exact_matches = []

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  for inputs_id, targets in dataloader:
      inputs_id = inputs_id.to(device)
      targets = targets.to(device)

      _, outputs = model(inputs_id)
      outputs = outputs[:,:-1]
      targets = targets[:, 1:]

      loss = loss_fn(outputs.permute(0, 2, 1), targets)

      outputs = outputs.argmax(-1)
      mask = (targets != PAD_ID)
      corrects = (outputs == targets) & mask
      accuracy = corrects.sum().item() / mask.sum().item()

      accuracies.append(accuracy)
      losses.append(loss.item())

        #exact match
      pad_mask = (targets == PAD_ID)
      token_ok = (outputs == targets) | pad_mask
      exact_per_sample = token_ok.all(dim=1)
      exact_match = exact_per_sample.float().mean().item()
      exact_matches.append(exact_match)


  loss = np.mean(losses)
  accuracy = np.mean(accuracies)
  exact_match = np.mean(exact_matches)

  return loss, accuracy, exact_match


In [ ]:
def train(model, dataloader, loss_fn, optimizer):

    model.train()

    losses = []
    accuracies = []
    exact_matches = []

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for inputs_id, targets in dataloader:
        inputs_id = inputs_id.to(device)
        targets = targets.to(device)

        _, outputs = model(inputs_id)
        outputs = outputs[:,:-1]
        targets = targets[:, 1:]

        loss = loss_fn(outputs.permute(0, 2, 1), targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        outputs = outputs.argmax(-1)
        mask = (targets != PAD_ID)
        corrects = (outputs == targets) & mask
        accuracy = corrects.sum().item() / mask.sum().item()

        #accuracies.append((outputs.argmax(-1) == targets).float().mean().item())
        #accuracy
        accuracies.append(accuracy)
        losses.append(loss.item())

        #exact match
        pad_mask = (targets == PAD_ID)
        token_ok = (outputs == targets) | pad_mask
        exact_per_sample = token_ok.all(dim=1)
        exact_match = exact_per_sample.float().mean().item()
        exact_matches.append(exact_match)


    loss = np.mean(losses)
    accuracy = np.mean(accuracies)
    exact_match = np.mean(exact_matches)

    return loss, accuracy, exact_match

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm

def train_rnn_model(model, trainDataSet, testDataSet, trainCommativefunction: bool = False,
                    num_epochs: int = 200, patience: int = 50, min_delta: float = 1e-4):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    print('Number of parameters:', sum(p.numel() for p in model.parameters()))

    optimizer = torch.optim.Adam(model.parameters(), lr=4e-4, weight_decay=1e-3)
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)

    if trainCommativefunction:
      train_loader = DataLoader(trainDataSet, batch_size=128,
                                collate_fn=collate_fn_ab_ba, shuffle=True)
    else:
      train_loader = DataLoader(trainDataSet, batch_size=128,
                                collate_fn=collate_fn, shuffle=True)

    test_loader  = DataLoader(testDataSet,  batch_size=128,
                              collate_fn=collate_fn, shuffle=False)

    best_val_loss = np.inf
    bad_epochs = 0

    for epoch in tqdm(range(1, num_epochs + 1)):
        if trainCommativefunction:
          train_loss, train_acc, train_em = train_commutative_function(model,
                                              train_loader, loss_fn, optimizer)
        else:
          train_loss, train_acc, train_em = train(model, train_loader, loss_fn, optimizer)

        val_loss, val_acc, val_em = evaluation(model, test_loader, loss_fn)

        if epoch % 50 == 0 or epoch == 1:
            print()
            print(f"[{epoch}] train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_em={train_em:.4f}")
            print(f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_em={val_em:.4f}")

        improved = (best_val_loss - val_loss) > min_delta
        if improved:
            best_val_loss = val_loss
            bad_epochs = 0
            torch.save(model.state_dict(), f"best_rnn_commloss_{trainCommativefunction}.pt")
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            print(f"Early stopping: val_loss does not improve {patience} epochs. "
                  f"Best val_loss={best_val_loss:.4f}")
            break

    model.load_state_dict(torch.load(f"best_rnn_commloss_{trainCommativefunction}.pt",
                                     map_location=device))
    return model

In [ ]:
trainDataSet = DatasetSum(train_dst.iloc[:,0].tolist())
testDataSet = DatasetSum(test_dst.iloc[:,0].tolist())

rnn_addition_simple = RNNAddition(len(symbol2id), hidden_size=256)
train_rnn_model(rnn_addition_simple, trainDataSet, testDataSet, num_epochs=500,
                                  trainCommativefunction=False)

Number of parameters: 138510


  0%|          | 1/500 [00:01<09:56,  1.19s/it]


[1] train_loss=1.7328 train_acc=0.3875 train_em=0.0009
val_loss=1.5544 val_acc=0.4133 val_em=0.0025


 10%|█         | 50/500 [01:03<09:20,  1.25s/it]


[50] train_loss=0.9258 train_acc=0.6596 train_em=0.0473
val_loss=0.9335 val_acc=0.6530 val_em=0.0437


...

 90%|█████████ | 450/500 [09:26<01:06,  1.32s/it]


[450] train_loss=0.0622 train_acc=0.9859 train_em=0.9429
val_loss=0.0856 val_acc=0.9723 val_em=0.8966


100%|██████████| 500/500 [10:28<00:00,  1.26s/it]


[500] train_loss=0.0552 train_acc=0.9885 train_em=0.9538
val_loss=0.0862 val_acc=0.9739 val_em=0.8996


RNNAddition(
  (emb_layer): Embedding(14, 256)
  (W): Linear(in_features=512, out_features=256, bias=True)
  (O): Linear(in_features=256, out_features=14, bias=True)
)

In [ ]:
torch.cuda.is_available()

True

In [ ]:
trainDataSet = DatasetSum(train_dst.iloc[:,0].tolist())
testDataSet = DatasetSum(test_dst.iloc[:,0].tolist())

rnn_addition_commloss = RNNAddition(len(symbol2id), hidden_size=256)
train_rnn_model(rnn_addition_commloss, trainDataSet, testDataSet, num_epochs=2000,
                                  trainCommativefunction=True)

Number of parameters: 138510


  0%|          | 1/2000 [00:02<1:35:33,  2.87s/it]


[1] train_loss=3.5095 train_acc=0.3856 train_em=0.0005
val_loss=1.5795 val_acc=0.4007 val_em=0.0010


  2%|▎         | 50/2000 [01:52<1:11:37,  2.20s/it]


[50] train_loss=1.8658 train_acc=0.6593 train_em=0.0460
val_loss=0.9298 val_acc=0.6559 val_em=0.0407


...


[1150] train_loss=0.0984 train_acc=0.9939 train_em=0.9739
val_loss=0.0628 val_acc=0.9809 val_em=0.9255


 60%|██████    | 1200/2000 [44:11<29:41,  2.23s/it]


[1200] train_loss=0.0954 train_acc=0.9941 train_em=0.9750
val_loss=0.0725 val_acc=0.9771 val_em=0.9097


 60%|██████    | 1204/2000 [44:20<29:19,  2.21s/it]


KeyboardInterrupt: 

In [ ]:
def generate_answer(model, example: str, max_new_tokens: int = 10):
    model.eval()
    device = next(model.parameters()).device

    prefix = torch.tensor(encode(example), dtype=torch.long, device=device).unsqueeze(0)
    with torch.no_grad():
        h_t, out = model(prefix)  # out: [1, seq_len, vocab]
        next_id = out[:, -1, :].argmax(-1).item() \

    generated = []
    with torch.no_grad():
        for _ in range(max_new_tokens):
            if next_id == EOS_ID:
                break
            generated.append(id2symbol[next_id])

            inp = torch.tensor([[next_id]], dtype=torch.long, device=device)
            h_t, out = model(inp, h_0=h_t)
            next_id = out[:, -1, :].argmax(-1).item()

    return "".join(generated)

In [ ]:
generate_answer(rnn_addition_simple, '1+3=')

'141'

In [ ]:
generate_answer(rnn_addition_simple, '1999+1024=')

'1125'

In [ ]:
generate_answer(rnn_addition_commloss, '100+300=')

'400'

In [ ]:
generate_answer(rnn_addition_commloss, '1999+1024=')

'831'

In [ ]:
dst_4d = pd.read_csv("data_4_19997.csv")
dst_4d.head()

,0
0,8510+8322=16832
1,4441+9516=13957
2,2147+1034=3181
3,4175+9425=13600
4,9320+5031=14351


In [ ]:
testDataSet4d = DatasetSum(dst_4d.iloc[:,0].tolist())

In [ ]:
print("Simple model")
print()
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)
test_loader  = DataLoader(testDataSet,  batch_size=128, collate_fn=collate_fn, shuffle=False)
loss, acc, em = evaluation(rnn_addition_simple, test_loader, loss_fn)
print("Tested on 1-3 digits dataset")
print("Loss: ", loss)
print("Accuracy: ", acc)
print("Exact match: ", em)
print()
print("Tested on 4 digits dataset")
test_loader_4d  = DataLoader(testDataSet4d,  batch_size=128, collate_fn=collate_fn, shuffle=False)
loss, acc, em = evaluation(rnn_addition_simple, test_loader_4d, loss_fn)
print("Loss: ", loss)
print("Accuracy: ", acc)
print("Exact match: ", em)

Simple model

Tested on 1-3 digits dataset
Loss:  0.07442402178722043
Accuracy:  0.9784959498731325
Exact match:  0.9205679335901814

Tested on 4 digits dataset
Loss:  7.198002991402984
Accuracy:  0.17661261406241452
Exact match:  0.0


In [ ]:
print("Commitative loss model")
print()
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)
test_loader  = DataLoader(testDataSet,  batch_size=128, collate_fn=collate_fn, shuffle=False)
loss, acc, em = evaluation(rnn_addition_commloss, test_loader, loss_fn)
print("Tested on 1-3 digits dataset")
print("Loss: ", loss)
print("Accuracy: ", acc)
print("Exact match: ", em)
print()
print("Tested on 4 digits dataset")
test_loader_4d  = DataLoader(testDataSet4d,  batch_size=128, collate_fn=collate_fn, shuffle=False)
loss, acc, em = evaluation(rnn_addition_commloss, test_loader_4d, loss_fn)
print("Loss: ", loss)
print("Accuracy: ", acc)
print("Exact match: ", em)

Commitative loss model

Tested on 1-3 digits dataset
Loss:  0.06826712779941098
Accuracy:  0.979403956735006
Exact match:  0.921934800763284

Tested on 4 digits dataset
Loss:  6.231433300455665
Accuracy:  0.06445742958215375
Exact match:  0.0
